# Top-20 Search For Uplift@10

Цель ноутбука: не заменить рабочий baseline, а проверить более сильные кандидаты для leaderboard.

Что проверяем:

- регуляризованный `Hurdle T-learner` на CatBoost;
- `Hurdle S-learner` как более гладкую альтернативу;
- full features против аккуратного удаления почти дублей;
- stratified validation и stress holdout по верхним `user_id`;
- оптимизацию по `uplift_lower_80ci`, потому что именно это похоже на публичный скор;
- финальные multi-seed ensemble и несколько submission-кандидатов.

Важно: если предыдущие rank-blend сабмиты ухудшили public score, не надо просто усиливать веса. Здесь мы ищем новый устойчивый сигнал через регуляризацию и проверку feature set.

In [ ]:
import gc
import json
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from IPython.display import display
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Modes:
# - smoke: quick check that everything runs
# - lowmem: stable VM run when kernels die from memory pressure
# - vm: stronger search on a VM with enough RAM
# - full: wider search if there is enough time
RUN_MODE = os.environ.get("UPLIFT_RUN_MODE", "smoke").lower()
assert RUN_MODE in {"smoke", "lowmem", "vm", "full"}
CATBOOST_THREAD_COUNT = int(os.environ.get("CATBOOST_THREAD_COUNT", max(1, (os.cpu_count() or 2) // 2)))

MODE_CFG = {
    "smoke": {
        "n_configs": 3,
        "iterations": 450,
        "early_stopping_rounds": 80,
        "bootstrap_iters": 80,
        "stress_top_n": 2,
        "final_top_n": 1,
        "final_seeds": [42],
        "sample_train_n": None,
    },
    "lowmem": {
        "n_configs": 8,
        "iterations": 900,
        "early_stopping_rounds": 100,
        "bootstrap_iters": 120,
        "stress_top_n": 3,
        "final_top_n": 2,
        "final_seeds": [42, 202],
        "sample_train_n": 240_000,
    },
    "vm": {
        "n_configs": 18,
        "iterations": 1300,
        "early_stopping_rounds": 160,
        "bootstrap_iters": 220,
        "stress_top_n": 5,
        "final_top_n": 3,
        "final_seeds": [42, 202],
        "sample_train_n": None,
    },
    "full": {
        "n_configs": 55,
        "iterations": 2400,
        "early_stopping_rounds": 220,
        "bootstrap_iters": 350,
        "stress_top_n": 14,
        "final_top_n": 5,
        "final_seeds": [42, 202, 777, 1001, 3407],
        "sample_train_n": None,
    },
}
CFG = MODE_CFG[RUN_MODE]
print("RUN_MODE:", RUN_MODE)
print("CATBOOST_THREAD_COUNT:", CATBOOST_THREAD_COUNT)
print(json.dumps(CFG, indent=2))

DATA_DIR = Path("кей")
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"
ARTIFACT_DIR = Path("top20_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

ID_COL = "user_id"
TARGET_COL = "rec_spend"
TREATMENT_COL = "treatment_flg"
COMM_COL = "communication_type"
TOP_K = 0.10

## Load Data

In [ ]:
train_raw = pd.read_parquet(TRAIN_PATH)
test_raw = pd.read_parquet(TEST_PATH)

if CFG["sample_train_n"] is not None and CFG["sample_train_n"] < len(train_raw):
    train_raw = train_raw.sample(CFG["sample_train_n"], random_state=RANDOM_STATE).sort_index().reset_index(drop=True)

print("train:", train_raw.shape)
print("test :", test_raw.shape)
print("target zero share:", float((train_raw[TARGET_COL] == 0).mean()))
print("test user_id range:", int(test_raw[ID_COL].min()), int(test_raw[ID_COL].max()))

display(train_raw[[TREATMENT_COL, COMM_COL, TARGET_COL]].groupby([COMM_COL, TREATMENT_COL])[TARGET_COL].agg(["size", "mean", "median"]))

## Metric

In [ ]:
def uplift_at_k(y_true, treatment, score, k=TOP_K):
    y_true = np.asarray(y_true, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    score = np.asarray(score, dtype=float)
    n_top = max(1, int(len(score) * k))
    top_idx = np.argsort(score)[-n_top:]
    top_treat = treatment[top_idx] == 1
    top_ctrl = treatment[top_idx] == 0
    if top_treat.sum() == 0 or top_ctrl.sum() == 0:
        return np.nan
    return float(y_true[top_idx][top_treat].mean() - y_true[top_idx][top_ctrl].mean())


def bootstrap_uplift_ci(y_true, treatment, score, k=TOP_K, n_iter=200, ci=0.80, seed=RANDOM_STATE):
    y_true = np.asarray(y_true, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    score = np.asarray(score, dtype=float)
    rng = np.random.default_rng(seed)
    n_top = max(1, int(len(score) * k))
    top_idx = np.argsort(score)[-n_top:]
    vals = []
    for _ in range(n_iter):
        idx = rng.choice(top_idx, size=n_top, replace=True)
        treat = treatment[idx] == 1
        ctrl = treatment[idx] == 0
        if treat.sum() == 0 or ctrl.sum() == 0:
            continue
        vals.append(y_true[idx][treat].mean() - y_true[idx][ctrl].mean())
    vals = np.asarray(vals, dtype=float)
    if len(vals) == 0:
        return {"bootstrap_mean": np.nan, "bootstrap_std": np.nan, "lower80": np.nan, "upper80": np.nan, "iters_used": 0}
    alpha = (1.0 - ci) / 2.0
    return {
        "bootstrap_mean": float(vals.mean()),
        "bootstrap_std": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
        "lower80": float(np.quantile(vals, alpha)),
        "upper80": float(np.quantile(vals, 1.0 - alpha)),
        "iters_used": int(len(vals)),
    }


def evaluate_scores(df, score, label, bootstrap_iters=None):
    bootstrap_iters = CFG["bootstrap_iters"] if bootstrap_iters is None else bootstrap_iters
    y = df[TARGET_COL].to_numpy()
    t = df[TREATMENT_COL].to_numpy()
    point = uplift_at_k(y, t, score)
    ci = bootstrap_uplift_ci(y, t, score, n_iter=bootstrap_iters)
    n_top = max(1, int(len(score) * TOP_K))
    top_idx = np.argsort(np.asarray(score))[-n_top:]
    top = df.iloc[top_idx]
    out = {
        "model": label,
        "n_eval": len(df),
        "n_top": n_top,
        "uplift_at10": point,
        "lower80": ci["lower80"],
        "upper80": ci["upper80"],
        "bootstrap_std": ci["bootstrap_std"],
        "top_target_mean": float(top[TARGET_COL].mean()),
        "top_treatment_share": float(top[TREATMENT_COL].mean()),
    }
    if COMM_COL in top.columns:
        for comm, share in top[COMM_COL].value_counts(normalize=True).sort_index().items():
            out[f"top_{comm}_share"] = float(share)
    return out


def rank_average(*scores, weights=None):
    arrs = [np.asarray(s, dtype=float) for s in scores]
    if weights is None:
        weights = np.ones(len(arrs), dtype=float) / len(arrs)
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    out = np.zeros_like(arrs[0], dtype=float)
    for s, w in zip(arrs, weights):
        out += w * pd.Series(s).rank(method="average").to_numpy()
    return out


def save_submission(test_df, scores, path):
    sub = pd.DataFrame({ID_COL: test_df[ID_COL].to_numpy(), "UPLIFT_SCORE": np.asarray(scores, dtype=float)})
    assert len(sub) == len(test_df)
    assert sub[ID_COL].equals(test_df[ID_COL].reset_index(drop=True))
    assert sub["UPLIFT_SCORE"].notna().all()
    sub.to_csv(path, index=False)
    print("saved", path, sub.shape)
    return sub

## Feature Engineering

In [ ]:
def add_engineered_features(df, base_missing_cols=None):
    out = df.copy()

    # A few stable ratios. They are guarded by column existence so the notebook survives schema changes.
    ratio_specs = [
        ("mark_view_per_offer", "cus_mark_n_view", "cus_mark_n_offers"),
        ("mark_view_per_rule", "cus_mark_n_view", "cus_mark_n_rule"),
        ("offers_per_rule", "cus_mark_n_offers", "cus_mark_n_rule"),
        ("rto_per_trn", "rto", "n_trn"),
        ("days_per_trn", "n_days_last_visit", "n_trn"),
        ("sku_per_trn", "n_sku", "n_trn"),
        ("cat7_per_sku", "n_cat_7", "n_sku"),
    ]
    for new_col, num, den in ratio_specs:
        if num in out.columns and den in out.columns:
            out[new_col] = out[num].astype(float) / (out[den].astype(float).abs() + 1.0)

    # Missingness itself can be signal in this task. CatBoost handles NaN, but explicit flags sometimes help ranking.
    if base_missing_cols is not None:
        for col in base_missing_cols:
            if col in out.columns:
                out[f"{col}__isna"] = out[col].isna().astype("int8")

    return out

# Choose missingness flags from train+test, but avoid exploding the feature space.
base_feature_cols = [c for c in test_raw.columns if c != ID_COL]
miss_rate = pd.concat([train_raw[base_feature_cols], test_raw[base_feature_cols]], axis=0).isna().mean().sort_values(ascending=False)
MISSING_FLAG_COLS = miss_rate[(miss_rate >= 0.03) & (miss_rate < 0.98)].head(35).index.tolist()
print("missing flag cols:", len(MISSING_FLAG_COLS))
print(MISSING_FLAG_COLS[:20])

train = add_engineered_features(train_raw, MISSING_FLAG_COLS)
test = add_engineered_features(test_raw, MISSING_FLAG_COLS)

FULL_FEATURES = [c for c in test.columns if c != ID_COL]
print("full features:", len(FULL_FEATURES))

In [ ]:
def make_pruned_features(train_df, test_df, features, threshold=0.985, sample_n=180_000):
    """Drop only near-duplicate numeric columns. This is an experiment, not the default truth."""
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(train_df[c])]
    if len(numeric) == 0:
        return features, []

    both = pd.concat([train_df[numeric], test_df[numeric]], axis=0)
    if len(both) > sample_n:
        both = both.sample(sample_n, random_state=RANDOM_STATE)
    corr = both.corr(method="spearman").abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    # Prefer keeping columns with fewer NaNs and higher variance.
    null_rate = both.isna().mean()
    variance = both.var(numeric_only=True).fillna(0)
    drop = set()
    for col in upper.columns:
        partners = upper.index[upper[col] > threshold].tolist()
        for partner in partners:
            if col in drop or partner in drop:
                continue
            score_col = (1.0 - null_rate.get(col, 1.0), variance.get(col, 0.0))
            score_partner = (1.0 - null_rate.get(partner, 1.0), variance.get(partner, 0.0))
            if score_col >= score_partner:
                drop.add(partner)
            else:
                drop.add(col)

    pruned = [c for c in features if c not in drop]
    return pruned, sorted(drop)

PRUNED_FEATURES, DROPPED_COLS = make_pruned_features(train, test, FULL_FEATURES)
print("pruned features:", len(PRUNED_FEATURES), "dropped:", len(DROPPED_COLS))
print(DROPPED_COLS[:60])

FEATURE_SETS = {
    "full": FULL_FEATURES,
    "pruned_corr985": PRUNED_FEATURES,
}

## Splits

In [ ]:
def make_strata(df):
    return df[TREATMENT_COL].astype(str) + "__" + df[COMM_COL].astype(str)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
strata = make_strata(train)
strat_train_idx, strat_val_idx = next(sss.split(train, strata))

# Stress holdout: test goes after train by user_id, so this checks extrapolation to newer/larger ids.
id_cutoff = train[ID_COL].quantile(0.75) if ID_COL in train.columns else train.index.to_series().quantile(0.75)
stress_val_mask = train[ID_COL] >= id_cutoff if ID_COL in train.columns else train.index >= id_cutoff
stress_train_idx = np.where(~stress_val_mask)[0]
stress_val_idx = np.where(stress_val_mask)[0]

SPLITS = {
    "strat": (strat_train_idx, strat_val_idx),
    "stress_user_id": (stress_train_idx, stress_val_idx),
}

for name, (_, val_idx) in SPLITS.items():
    part = train.iloc[val_idx]
    print(name, part.shape)
    display(part[[TREATMENT_COL, COMM_COL, TARGET_COL]].groupby([COMM_COL, TREATMENT_COL])[TARGET_COL].agg(["size", "mean"]))

## CatBoost Helpers

In [ ]:
def categorical_features(df, features):
    cats = []
    for c in features:
        if c not in df.columns:
            continue
        if (
            pd.api.types.is_object_dtype(df[c])
            or pd.api.types.is_string_dtype(df[c])
            or pd.api.types.is_categorical_dtype(df[c])
            or pd.api.types.is_bool_dtype(df[c])
        ):
            cats.append(c)
    return cats


def prepare_X(df, features):
    X = df[features].copy()
    cats = categorical_features(X, features)
    for c in cats:
        X[c] = X[c].astype("string").fillna("__MISSING__")
    return X


def common_catboost_params(config, seed):
    return {
        "iterations": int(config.get("iterations", CFG["iterations"])),
        "learning_rate": float(config["learning_rate"]),
        "depth": int(config["depth"]),
        "l2_leaf_reg": float(config["l2_leaf_reg"]),
        "random_strength": float(config["random_strength"]),
        "rsm": float(config["rsm"]),
        "bagging_temperature": float(config["bagging_temperature"]),
        "bootstrap_type": "Bayesian",
        "min_data_in_leaf": int(config["min_data_in_leaf"]),
        "border_count": int(config["border_count"]),
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "allow_writing_files": False,
        "random_seed": int(seed),
        "thread_count": CATBOOST_THREAD_COUNT,
        "verbose": False,
    }


def fit_hurdle_t_learner(train_df, features, config, seed=RANDOM_STATE, valid_df=None):
    X_all = prepare_X(train_df, features)
    cat_features = categorical_features(X_all, features)
    models = {}

    for arm in [0, 1]:
        mask = train_df[TREATMENT_COL].to_numpy() == arm
        arm_df = train_df.loc[mask]
        X_arm = X_all.loc[mask]
        y_nonzero = (arm_df[TARGET_COL].to_numpy() > 0).astype(int)

        clf_params = common_catboost_params(config, seed + 11 + arm)
        clf_params.update({"loss_function": "Logloss", "eval_metric": "AUC"})
        clf = CatBoostClassifier(**clf_params)

        eval_set = None
        use_best_model = False
        if valid_df is not None:
            vmask = valid_df[TREATMENT_COL].to_numpy() == arm
            if vmask.sum() > 100 and valid_df.loc[vmask, TARGET_COL].gt(0).nunique() == 2:
                X_valid = prepare_X(valid_df.loc[vmask], features)
                y_valid = (valid_df.loc[vmask, TARGET_COL].to_numpy() > 0).astype(int)
                eval_set = (X_valid, y_valid)
                use_best_model = True
        clf.fit(
            X_arm,
            y_nonzero,
            cat_features=cat_features,
            eval_set=eval_set,
            use_best_model=use_best_model,
            early_stopping_rounds=CFG["early_stopping_rounds"] if eval_set is not None else None,
        )

        pos_mask = mask & train_df[TARGET_COL].gt(0).to_numpy()
        reg_df = train_df.loc[pos_mask]
        X_pos = X_all.loc[pos_mask]
        y_pos = np.log1p(reg_df[TARGET_COL].to_numpy(dtype=float))

        reg_params = common_catboost_params(config, seed + 101 + arm)
        reg = CatBoostRegressor(**reg_params)
        eval_set_reg = None
        use_best_model_reg = False
        if valid_df is not None:
            vmask = (valid_df[TREATMENT_COL].to_numpy() == arm) & valid_df[TARGET_COL].gt(0).to_numpy()
            if vmask.sum() > 100:
                X_valid_pos = prepare_X(valid_df.loc[vmask], features)
                y_valid_pos = np.log1p(valid_df.loc[vmask, TARGET_COL].to_numpy(dtype=float))
                eval_set_reg = (X_valid_pos, y_valid_pos)
                use_best_model_reg = True
        reg.fit(
            X_pos,
            y_pos,
            cat_features=cat_features,
            eval_set=eval_set_reg,
            use_best_model=use_best_model_reg,
            early_stopping_rounds=CFG["early_stopping_rounds"] if eval_set_reg is not None else None,
        )
        models[arm] = {"clf": clf, "reg": reg}
        gc.collect()

    return {"kind": "hurdle_t", "models": models, "features": list(features), "config": dict(config)}


def predict_hurdle_t_learner(model_pack, df):
    features = model_pack["features"]
    X = prepare_X(df, features)
    expected = {}
    for arm in [0, 1]:
        clf = model_pack["models"][arm]["clf"]
        reg = model_pack["models"][arm]["reg"]
        p = clf.predict_proba(X)[:, 1]
        amount = np.expm1(reg.predict(X))
        amount = np.clip(amount, 0, np.nanpercentile(train[TARGET_COL].to_numpy(dtype=float), 99.9))
        expected[arm] = p * amount
    return expected[1] - expected[0]

In [ ]:
def fit_hurdle_s_learner(train_df, features, config, seed=RANDOM_STATE, valid_df=None):
    s_features = list(features) + ["__treatment_model"]
    work = train_df.copy()
    work["__treatment_model"] = work[TREATMENT_COL].astype("int8")
    X_all = prepare_X(work, s_features)
    cat_features = categorical_features(X_all, s_features)
    y_nonzero = work[TARGET_COL].gt(0).astype(int).to_numpy()

    clf_params = common_catboost_params(config, seed + 501)
    clf_params.update({"loss_function": "Logloss", "eval_metric": "AUC"})
    clf = CatBoostClassifier(**clf_params)

    eval_set = None
    use_best_model = False
    if valid_df is not None:
        valid_work = valid_df.copy()
        valid_work["__treatment_model"] = valid_work[TREATMENT_COL].astype("int8")
        X_valid = prepare_X(valid_work, s_features)
        y_valid = valid_work[TARGET_COL].gt(0).astype(int).to_numpy()
        eval_set = (X_valid, y_valid)
        use_best_model = True
    clf.fit(
        X_all,
        y_nonzero,
        cat_features=cat_features,
        eval_set=eval_set,
        use_best_model=use_best_model,
        early_stopping_rounds=CFG["early_stopping_rounds"] if eval_set is not None else None,
    )

    pos = work[TARGET_COL].gt(0)
    reg_params = common_catboost_params(config, seed + 601)
    reg = CatBoostRegressor(**reg_params)
    X_pos = X_all.loc[pos]
    y_pos = np.log1p(work.loc[pos, TARGET_COL].to_numpy(dtype=float))

    eval_set_reg = None
    use_best_model_reg = False
    if valid_df is not None:
        valid_work = valid_df.copy()
        valid_work["__treatment_model"] = valid_work[TREATMENT_COL].astype("int8")
        vpos = valid_work[TARGET_COL].gt(0)
        if vpos.sum() > 100:
            X_valid_pos = prepare_X(valid_work.loc[vpos], s_features)
            y_valid_pos = np.log1p(valid_work.loc[vpos, TARGET_COL].to_numpy(dtype=float))
            eval_set_reg = (X_valid_pos, y_valid_pos)
            use_best_model_reg = True
    reg.fit(
        X_pos,
        y_pos,
        cat_features=cat_features,
        eval_set=eval_set_reg,
        use_best_model=use_best_model_reg,
        early_stopping_rounds=CFG["early_stopping_rounds"] if eval_set_reg is not None else None,
    )
    return {"kind": "hurdle_s", "clf": clf, "reg": reg, "features": s_features, "base_features": list(features), "config": dict(config)}


def predict_hurdle_s_learner(model_pack, df):
    expected = {}
    for arm in [0, 1]:
        work = df.copy()
        work["__treatment_model"] = arm
        X = prepare_X(work, model_pack["features"])
        p = model_pack["clf"].predict_proba(X)[:, 1]
        amount = np.expm1(model_pack["reg"].predict(X))
        amount = np.clip(amount, 0, np.nanpercentile(train[TARGET_COL].to_numpy(dtype=float), 99.9))
        expected[arm] = p * amount
    return expected[1] - expected[0]

## Hyperparameter Search

In [ ]:
def sample_regularized_config(rng, idx):
    # Deliberately regularized space. This is aimed at stable top-10 ranking, not max train fit.
    return {
        "config_id": int(idx),
        "iterations": CFG["iterations"],
        "learning_rate": float(rng.choice([0.025, 0.035, 0.05, 0.07])),
        "depth": int(rng.choice([4, 5, 6])),
        "l2_leaf_reg": float(rng.choice([12, 20, 35, 55, 85, 130])),
        "random_strength": float(rng.choice([1.0, 3.0, 6.0, 10.0, 16.0])),
        "rsm": float(rng.choice([0.65, 0.75, 0.85, 0.95])),
        "bagging_temperature": float(rng.choice([0.5, 1.0, 2.0, 4.0, 7.0])),
        "min_data_in_leaf": int(rng.choice([50, 100, 200, 350, 600])),
        "border_count": int(rng.choice([64, 128, 254])),
    }


def fit_predict_kind(kind, train_part, val_part, features, config, seed):
    if kind == "hurdle_t":
        model = fit_hurdle_t_learner(train_part, features, config, seed=seed, valid_df=val_part)
        return predict_hurdle_t_learner(model, val_part), model
    if kind == "hurdle_s":
        model = fit_hurdle_s_learner(train_part, features, config, seed=seed, valid_df=val_part)
        return predict_hurdle_s_learner(model, val_part), model
    raise ValueError(kind)


def local_objective(row):
    # lower80 dominates because leaderboard seems to use public lower CI.
    # point metric remains as a tie-breaker so we do not choose ultra-conservative weak models.
    return 0.78 * row["lower80"] + 0.22 * row["uplift_at10"]

rng = np.random.default_rng(RANDOM_STATE)
search_rows = []
configs = [sample_regularized_config(rng, i) for i in range(CFG["n_configs"])]

# Phase 1: broad search on stratified validation. Include pruned feature set only on part of configs to save time.
for cfg_i, config in enumerate(configs, start=1):
    for feature_set_name in (["full", "pruned_corr985"] if cfg_i <= max(4, CFG["n_configs"] // 3) else ["full"]):
        for kind in (["hurdle_t", "hurdle_s"] if cfg_i <= max(3, CFG["n_configs"] // 4) else ["hurdle_t"]):
            split_name = "strat"
            tr_idx, va_idx = SPLITS[split_name]
            train_part = train.iloc[tr_idx].reset_index(drop=True)
            val_part = train.iloc[va_idx].reset_index(drop=True)
            features = FEATURE_SETS[feature_set_name]
            label = f"{kind}_{feature_set_name}_cfg{config['config_id']:03d}_{split_name}"
            print("fit", label)
            start = time.time()
            scores, _ = fit_predict_kind(kind, train_part, val_part, features, config, seed=RANDOM_STATE + config["config_id"] * 10)
            metrics = evaluate_scores(val_part, scores, label)
            metrics.update({
                "kind": kind,
                "feature_set": feature_set_name,
                "split": split_name,
                "config_id": config["config_id"],
                "elapsed_min": (time.time() - start) / 60,
                **{f"param_{k}": v for k, v in config.items() if k != "config_id"},
            })
            metrics["objective"] = local_objective(metrics)
            search_rows.append(metrics)
            display(pd.DataFrame(search_rows).sort_values("objective", ascending=False).head(10))
            pd.DataFrame(search_rows).to_csv(ARTIFACT_DIR / "top20_phase1_results.csv", index=False)
            gc.collect()

phase1 = pd.DataFrame(search_rows).sort_values("objective", ascending=False).reset_index(drop=True)
display(phase1.head(20))

## Stress Recheck By User ID

In [ ]:
stress_rows = []
phase1_top = phase1.head(CFG["stress_top_n"]).copy()

for _, best in phase1_top.iterrows():
    config = {k.replace("param_", ""): best[k] for k in best.index if k.startswith("param_")}
    config["config_id"] = int(best["config_id"])
    for int_key in ["iterations", "depth", "min_data_in_leaf", "border_count"]:
        config[int_key] = int(config[int_key])
    for float_key in ["learning_rate", "l2_leaf_reg", "random_strength", "rsm", "bagging_temperature"]:
        config[float_key] = float(config[float_key])

    split_name = "stress_user_id"
    tr_idx, va_idx = SPLITS[split_name]
    train_part = train.iloc[tr_idx].reset_index(drop=True)
    val_part = train.iloc[va_idx].reset_index(drop=True)
    features = FEATURE_SETS[best["feature_set"]]
    label = f"{best['kind']}_{best['feature_set']}_cfg{int(best['config_id']):03d}_{split_name}"
    print("stress fit", label)
    start = time.time()
    scores, _ = fit_predict_kind(best["kind"], train_part, val_part, features, config, seed=RANDOM_STATE + int(best["config_id"]) * 100)
    metrics = evaluate_scores(val_part, scores, label)
    metrics.update({
        "kind": best["kind"],
        "feature_set": best["feature_set"],
        "split": split_name,
        "config_id": int(best["config_id"]),
        "phase1_objective": float(best["objective"]),
        "elapsed_min": (time.time() - start) / 60,
        **{f"param_{k}": v for k, v in config.items() if k != "config_id"},
    })
    metrics["stress_objective"] = local_objective(metrics)
    stress_rows.append(metrics)
    pd.DataFrame(stress_rows).to_csv(ARTIFACT_DIR / "top20_stress_results.csv", index=False)
    display(pd.DataFrame(stress_rows).sort_values("stress_objective", ascending=False))
    gc.collect()

stress = pd.DataFrame(stress_rows)
merged = phase1.merge(
    stress[["kind", "feature_set", "config_id", "stress_objective", "lower80", "uplift_at10"]].rename(columns={"lower80": "stress_lower80", "uplift_at10": "stress_uplift_at10"}),
    on=["kind", "feature_set", "config_id"],
    how="left",
)
merged["combined_objective"] = (
    0.58 * merged["objective"]
    + 0.30 * merged["stress_objective"].fillna(merged["objective"])
    + 0.12 * np.minimum(merged["lower80"], merged["stress_lower80"].fillna(merged["lower80"]))
)
merged = merged.sort_values("combined_objective", ascending=False).reset_index(drop=True)
merged.to_csv(ARTIFACT_DIR / "top20_combined_results.csv", index=False)
display(merged.head(20))

best_configs = merged.head(CFG["final_top_n"])
best_configs.to_csv(ARTIFACT_DIR / "top20_selected_configs.csv", index=False)
print("selected configs:")
display(best_configs)

## Final Training And Submissions

In [ ]:
def row_to_config(row):
    config = {k.replace("param_", ""): row[k] for k in row.index if k.startswith("param_")}
    config["config_id"] = int(row["config_id"])
    for int_key in ["iterations", "depth", "min_data_in_leaf", "border_count"]:
        config[int_key] = int(config[int_key])
    for float_key in ["learning_rate", "l2_leaf_reg", "random_strength", "rsm", "bagging_temperature"]:
        config[float_key] = float(config[float_key])
    return config

final_scores = {}
final_score_parts = []

for row_i, row in best_configs.iterrows():
    config = row_to_config(row)
    features = FEATURE_SETS[row["feature_set"]]
    kind = row["kind"]
    cfg_name = f"{kind}_{row['feature_set']}_cfg{int(row['config_id']):03d}"
    seed_scores = []
    print("final candidate", cfg_name, json.dumps(config, indent=2))

    for seed in CFG["final_seeds"]:
        print("  seed", seed)
        if kind == "hurdle_t":
            model = fit_hurdle_t_learner(train, features, config, seed=seed, valid_df=None)
            pred = predict_hurdle_t_learner(model, test)
        elif kind == "hurdle_s":
            model = fit_hurdle_s_learner(train, features, config, seed=seed, valid_df=None)
            pred = predict_hurdle_s_learner(model, test)
        else:
            raise ValueError(kind)
        seed_scores.append(pred)
        final_score_parts.append((f"{cfg_name}_seed{seed}", pred))
        gc.collect()

    # Rank ensemble is usually safer for uplift top-k than raw averaging with different score scales.
    final_scores[cfg_name] = rank_average(*seed_scores)
    save_submission(test, final_scores[cfg_name], f"predictions_top20_{cfg_name}.csv")

# Family-level blends.
if final_scores:
    all_names = list(final_scores)
    final_scores["top20_rank_ensemble_all"] = rank_average(*[final_scores[n] for n in all_names])
    save_submission(test, final_scores["top20_rank_ensemble_all"], "predictions_top20_rank_ensemble_all.csv")

    hurdle_names = [n for n in all_names if n.startswith("hurdle_t")]
    if len(hurdle_names) >= 2:
        final_scores["top20_rank_ensemble_hurdle_t"] = rank_average(*[final_scores[n] for n in hurdle_names])
        save_submission(test, final_scores["top20_rank_ensemble_hurdle_t"], "predictions_top20_rank_ensemble_hurdle_t.csv")

print("final candidates:", list(final_scores))

## Conservative Communication Calibration Candidates

In [ ]:
# This is intentionally conservative. Earlier diagnostics showed the best public file already downweighted com_type_2,
# so strong manual channel priors can make the public score worse.
def communication_prior_from_train(train_df):
    pivot = train_df.groupby([COMM_COL, TREATMENT_COL])[TARGET_COL].mean().unstack()
    prior = (pivot[1] - pivot[0]).to_dict()
    return prior

comm_prior = communication_prior_from_train(train)
print("raw train uplift prior by communication_type:", comm_prior)
prior_values = test[COMM_COL].map(comm_prior).astype(float).to_numpy()
prior_rank = pd.Series(prior_values).rank(method="average").to_numpy()

calibrated_scores = {}
base_name = "top20_rank_ensemble_all" if "top20_rank_ensemble_all" in final_scores else next(iter(final_scores))
base_score = final_scores[base_name]

for alpha in [0.015, 0.03, 0.05]:
    calibrated = (1 - alpha) * pd.Series(base_score).rank(method="average").to_numpy() + alpha * prior_rank
    name = f"top20_{base_name}_commcal_a{str(alpha).replace('.', '')}"
    calibrated_scores[name] = calibrated
    save_submission(test, calibrated, f"predictions_{name}.csv")

# Main local pick: no manual calibration by default. Submit this first unless local/public evidence says otherwise.
MAIN_SUBMISSION = base_name
save_submission(test, final_scores[MAIN_SUBMISSION], "predictions.csv")

summary = []
for name, scores in {**final_scores, **calibrated_scores}.items():
    top_idx = np.argsort(scores)[-int(len(scores) * TOP_K):]
    top = test.iloc[top_idx]
    row = {
        "candidate": name,
        "path": "predictions.csv" if name == MAIN_SUBMISSION else f"predictions_{name}.csv",
        "score_mean": float(np.mean(scores)),
        "score_std": float(np.std(scores)),
        "score_min": float(np.min(scores)),
        "score_max": float(np.max(scores)),
    }
    for comm, share in top[COMM_COL].value_counts(normalize=True).sort_index().items():
        row[f"top10_{comm}_share"] = float(share)
    summary.append(row)

summary_df = pd.DataFrame(summary).sort_values("candidate")
summary_df.to_csv(ARTIFACT_DIR / "top20_submission_summary.csv", index=False)
display(summary_df)
print("MAIN_SUBMISSION:", MAIN_SUBMISSION, "-> predictions.csv")

## What To Submit First

Порядок попыток, если лимит сабмитов маленький:

1. `predictions.csv` из этого ноутбука: это лучший локальный multi-seed rank ensemble без ручного перекоса каналов.
2. Если первый результат около/выше текущего public best, пробовать `predictions_top20_rank_ensemble_all.csv` явно, если он отличается от `predictions.csv`.
3. `predictions_top20_*_commcal_a0015.csv` только если локальный summary показывает, что top-10 почти не меняется. Большие `alpha` отправлять осторожно: предыдущий опыт показал, что ручные веса могут ухудшать public.

Что смотреть после прогона:

- `top20_artifacts/top20_combined_results.csv` — какие гиперпараметры победили;
- `top20_artifacts/top20_submission_summary.csv` — состав top-10 по `communication_type`;
- `top20_artifacts/top20_selected_configs.csv` — конфиги, которые можно потом зафиксировать и запускать без поиска.